Daten Laden und Spaltennamen ausgeben lassen

In [3]:
# import pandas and set display options
# define the header to see all columns in the dataframe

import pandas as pd
pd.set_option("display.max_columns", None)

pfad = "../data/en.openfoodfacts.org.products.csv.gz"   # an deinen Dateinamen anpassen

header = pd.read_csv(pfad, sep="\t", nrows=0)  # nur die Spaltennamen einlesen


In [4]:
# display the column names of the dataframe
# evaluate the relevance of the columns for the analysis and select the relevant ones


for spalte in header.columns:
    print(spalte)

code
url
creator
created_t
created_datetime
last_modified_t
last_modified_datetime
last_modified_by
last_updated_t
last_updated_datetime
product_name
abbreviated_product_name
generic_name
quantity
packaging
packaging_tags
packaging_en
packaging_text
brands
brands_tags
brands_en
categories
categories_tags
categories_en
origins
origins_tags
origins_en
manufacturing_places
manufacturing_places_tags
labels
labels_tags
labels_en
emb_codes
emb_codes_tags
first_packaging_code_geo
cities
cities_tags
purchase_places
stores
countries
countries_tags
countries_en
ingredients_text
ingredients_tags
ingredients_analysis_tags
allergens
allergens_en
traces
traces_tags
traces_en
serving_size
serving_quantity
no_nutrition_data
additives_n
additives
additives_tags
additives_en
nutriscore_score
nutriscore_grade
nova_group
pnns_groups_1
pnns_groups_2
food_groups
food_groups_tags
food_groups_en
states
states_tags
states_en
brand_owner
environmental_score_score
environmental_score_grade
nutrient_levels_tags
p

In [5]:
#define the relevant columns for the analysis


relevante_spalten = [
    # Identifikation & Kategorien
    "code",
    "product_name",
    "brands",
    "countries_tags",
    "main_category_en",
    "pnns_groups_1",
    "pnns_groups_2",
    "ingredients_analysis_tags",
    "labels_tags",
    # Scores & Verarbeitung
    "nutriscore_score",
    "nutriscore_grade",
    "nova_group",
    "additives_n",
    "environmental_score_score",
    "environmental_score_grade",
    # Hauptnährwerte (EU-Pflichtangaben)
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "fruits-vegetables-legumes_100g",
    # Datenqualität & Popularität
    "completeness",
    "unique_scans_n",
    "data_quality_errors_tags",
    "last_modified_datetime",
]

In [6]:
#read the CSV file in chunks to avoid memory issues


chunks = []
chunk_groesse = 100_000

for i, chunk in enumerate(
    pd.read_csv(
        pfad,
        sep="\t",
        usecols=relevante_spalten,
        chunksize=chunk_groesse,
        on_bad_lines="skip",
        low_memory=True,
    )
):
    # nur Produkte aus Deutschland behalten
    chunk = chunk[chunk["countries_tags"].str.contains("en:germany", na=False)]

    chunks.append(chunk)
    if (i + 1) % 5 == 0:
        print(f"{(i + 1) * chunk_groesse:,} Zeilen verarbeitet...")

df = pd.concat(chunks, ignore_index=True)
print(f"Fertig! Geladene Zeilen: {len(df):,}")

/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_27604/368029685.py:7: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


500,000 Zeilen verarbeitet...
1,000,000 Zeilen verarbeitet...
1,500,000 Zeilen verarbeitet...
2,000,000 Zeilen verarbeitet...
2,500,000 Zeilen verarbeitet...
3,000,000 Zeilen verarbeitet...
3,500,000 Zeilen verarbeitet...
4,000,000 Zeilen verarbeitet...


/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_27604/368029685.py:7: DtypeWarning: Columns (0: code, 44: ingredients_analysis_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


4,500,000 Zeilen verarbeitet...
5,000,000 Zeilen verarbeitet...
Fertig! Geladene Zeilen: 439,705


In [7]:
#display the first 10 rows of the dataframe


df.head(10)

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g
0,204286484,2026-03-26T02:25:06Z,Mehrkomponeneten Protein 90 C6 Haselnuß,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.000000
1,250632969,2026-03-26T02:25:07Z,Mehrkomponeneten Protein 90 C6 Banane,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.000000
2,460938714,2026-03-26T02:25:08Z,100% Soja Protein Haselnuss,allfitnessfactory.de,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.8625,Protein powders,362.9,0.5,NaN,1.1,NaN,NaN,88.50,NaN,0.000000
3,470322800,2026-03-26T02:25:09Z,Whey Protein aus Molke Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
4,501050603,2026-03-26T02:25:10Z,Whey Protein aus Molke 1000 Gramm Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
5,526938306,2026-03-26T02:25:12Z,Whey Protein aus Molke 500 Gramm Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
6,790310013,2026-03-26T02:25:12Z,Sour Fruit Gummies,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:non-vegetarian",9.0,19.0,e,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,325.0,0.0,0.0,75.0,57.5,0.0,5.00,0.12700,11.363636
7,790310020,2026-03-26T02:25:12Z,Jelly Fish,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",7.0,7.0,c,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,140.0,0.0,0.0,34.0,24.0,0.0,0.00,0.07620,0.000000
8,1100000136,2018-08-15T07:17:53Z,helle Soße,NaN,NaN,"en:france,en:germany",NaN,NaN,NaN,unknown,NaN,unknown,unknown,NaN,unknown,NaN,NaN,0.2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1182000086,2020-02-01T11:25:20Z,Salat der Saison Dressing für Feldsalat,NaN,NaN,en:germany,NaN,NaN,NaN,unknown,NaN,Fat and sauces,Dressings and sauces,37.0,d,NaN,NaN,0.3000,en:groceries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
#check on the shape of the dataframe

df.shape

(439705, 28)

In [9]:
#check the number of missing values in the dataframe

df.isna().sum()

code                                   0
last_modified_datetime                 0
product_name                       20766
brands                             86554
labels_tags                       312956
countries_tags                         0
ingredients_analysis_tags         295118
additives_n                       306594
nutriscore_score                  311960
nutriscore_grade                    1750
nova_group                        323006
pnns_groups_1                       1710
pnns_groups_2                       1710
environmental_score_score         340341
environmental_score_grade           1720
data_quality_errors_tags          430888
unique_scans_n                    221785
completeness                          26
main_category_en                  250320
energy-kcal_100g                  190932
fat_100g                          194376
saturated-fat_100g                211665
carbohydrates_100g                193482
sugars_100g                       203522
fiber_100g      

In [10]:
# filter the dataframe to only include products with a completeness of at least 0.7, no known data quality errors, and a non-null product name
df_clean = df[
    (df["completeness"] >= 0.7)
    & (df["data_quality_errors_tags"].isna())  # nur Produkte ohne bekannte Datenfehler
    & (df["product_name"].notna())
].copy()

In [11]:
# control how many null values have remained
df_clean.isna().sum()

code                                  0
last_modified_datetime                0
product_name                          0
brands                              287
labels_tags                       22165
countries_tags                        0
ingredients_analysis_tags          1857
additives_n                        1980
nutriscore_score                  10074
nutriscore_grade                     38
nova_group                         7197
pnns_groups_1                        37
pnns_groups_2                        37
environmental_score_score         29141
environmental_score_grade            38
data_quality_errors_tags          74657
unique_scans_n                     5479
completeness                          0
main_category_en                    259
energy-kcal_100g                   3318
fat_100g                           3593
saturated-fat_100g                 4670
carbohydrates_100g                 3501
sugars_100g                        3994
fiber_100g                        26632


In [ ]:
df_clean.duplicated().sum()

np.int64(0)

In [15]:
# control if there are unknown values in the columns "nutriscore_grade" and "pnns_groups_1"
print(df_clean["nutriscore_grade"].value_counts(dropna=False))
print(df_clean["pnns_groups_1"].value_counts(dropna=False))

nutriscore_grade
e                 16513
d                 15960
c                 15553
a                 10318
b                  6239
unknown            5971
not-applicable     4065
NaN                  38
Name: count, dtype: int64
pnns_groups_1
Sugary snacks              11214
Cereals and potatoes       10347
Milk and dairy products     9824
unknown                     9112
Beverages                   7711
Fish Meat Eggs              7163
Fruits and vegetables       5102
Fat and sauces              5099
Composite foods             4362
Salty snacks                3071
Alcoholic beverages         1431
Baby foods                   184
NaN                           37
Name: count, dtype: int64


In [19]:
df_clean[df_clean["nutriscore_grade"] == "unknown"].head(30)

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g
45,12200002005,2026-03-26T02:25:19Z,Chinese dumpling sheets,Master Lau,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",0.0,NaN,unknown,4.0,Cereals and potatoes,Cereals,NaN,unknown,NaN,NaN,0.7750,Pastas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
182,30000050408,2026-03-25T18:35:47Z,Aunt Jemima Original - Pancake & Waffle Mix,"Aunt Jemima, Aunt Jemima - The Quaker Oats Com...",NaN,"en:france,en:germany,en:united-states","en:palm-oil-free,en:vegan-status-unknown,en:ve...",3.0,NaN,unknown,3.0,Sugary snacks,Biscuits and cakes,NaN,unknown,NaN,1.0,0.7750,fr:préparation pancake,319.000000,1.060000,0.000000,70.210000,14.890000,2.100000,8.510000,3.935000,0.000000
262,40000497561,2026-03-14T09:42:04Z,peanut butter m&m's chocolate candies,M&M's,NaN,"en:austria,en:france,en:germany,en:united-states","en:palm-oil,en:non-vegan,en:vegetarian-status-...",9.0,NaN,unknown,4.0,Sugary snacks,Sweets,NaN,unknown,NaN,1.0,0.7625,Bonbons,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
269,41000022548,2026-06-10T09:12:53Z,Fettuccine Pasta in a Creamy Mushroom Flavored...,Knorr,en:no-artificial-flavors,"en:germany,en:united-states","en:may-contain-palm-oil,en:non-vegan,en:non-ve...",5.0,NaN,unknown,4.0,Cereals and potatoes,Cereals,NaN,unknown,NaN,1.0,0.7000,en:undefined,361.558405,1.322775,NaN,74.957230,3.527399,2.645549,13.227747,2.072347,5.780000
276,41321009754,2026-06-01T17:44:47Z,Balsamic Vinaigrette Dressing,Wish-Bone,"en:no-artificial-flavors,en:No HFCS","en:bolivia,en:france,en:germany,en:united-states","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",5.0,NaN,unknown,4.0,Fat and sauces,Dressings and sauces,42.0,d,NaN,8.0,0.7750,Vinaigrettes,200.000000,16.666667,3.333333,9.186000,13.333333,0.000000,0.000000,2.416667,5.833333
305,42572050007,2026-03-25T23:22:14Z,Original Non-Alcoholic Beer,Clausthaler,NaN,"en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,Beverages,Unsweetened beverages,41.0,d,NaN,1.0,0.7625,Non-alcoholic beers,26.000000,0.000000,NaN,5.633803,NaN,NaN,0.309859,NaN,16.666667
322,43000285213,2026-11-11T18:34:29Z,Stuffing Mix for Chicken,Stove Top,NaN,"en:germany,en:united-states","en:palm-oil-free,en:non-vegan,en:non-vegetarian",6.0,NaN,unknown,4.0,Cereals and potatoes,Cereals,NaN,unknown,NaN,2.0,0.8000,stuffing,110.000000,1.000000,NaN,21.000000,2.000000,1.000000,3.000000,0.975000,11.570000
355,48375014109,2026-03-25T22:07:03Z,Palmherzen,La Cima,"en:kosher,en:green-dot,en:organized-kashrut-ko...",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,unknown,unknown,80.0,a,NaN,NaN,0.8875,en:canned-hearts-of-palm,16.000000,0.000000,NaN,1.000000,NaN,NaN,3.000000,NaN,100.000000
367,50000334803,2026-08-07T10:23:05Z,Vanilla Caramel Zero Sugar Coffee Mate,Nestlé,"en:low-or-no-sugar,en:kosher,en:no-sugar,en:or...","en:france,en:germany,en:united-states","en:palm-oil,en:non-vegan,en:vegetarian-status-...",4.0,NaN,unknown,4.0,Beverages,Plant-based milk substitutes,NaN,unknown,NaN,3.0,0.8875,Creamer,750.000000,50.000000,40.000000,40.000000,0.000000,0.000000,0.000000,0.750000,5.940000
389,54467050351,2026-06-14T07:34:22Z,Hot cocoa double chocolate,Starbucks,en:pure-cocoa-butter,"en:germany,en:united-states","en:palm-oil-content-unknown,en:vegan-status-un...",0.0,NaN,unknown,NaN,Beverages,Unsweetened beverages,NaN,unknown,NaN,1.0,0.7000,Dehydrated beverages,393.000000,10.710000,5.360000,78.570000,60.710000,10.700000,7.140000,0.000000,0.000000


In [26]:
df_clean.loc[262,"ingredients_analysis_tags"]

'en:palm-oil,en:non-vegan,en:vegetarian-status-unknown'

In [20]:
#replace "unknown" and "not-applicable" with NaN in the entire DataFrame
#filter out any unplausible values in the nutritional columns based on physical plausibility limits per 100 g

import numpy as np


# Physikalische Plausibilitätsgrenzen pro 100 g prüfen & Ausreißer entfernen
# (Max. 100 g pro 100 g für Makronährstoffe, max. 900 kcal für reines Fett, max. 100 g Salz)
plausibel = (
    (
        df_clean["energy-kcal_100g"].isna()
        | df_clean["energy-kcal_100g"].between(0, 900)
    )
    & (df_clean["fat_100g"].isna() | df_clean["fat_100g"].between(0, 100))
    & (
        df_clean["saturated-fat_100g"].isna()
        | df_clean["saturated-fat_100g"].between(0, 100)
    )
    & (
        df_clean["carbohydrates_100g"].isna()
        | df_clean["carbohydrates_100g"].between(0, 100)
    )
    & (df_clean["sugars_100g"].isna() | df_clean["sugars_100g"].between(0, 100))
    & (df_clean["fiber_100g"].isna() | df_clean["fiber_100g"].between(0, 100))
    & (
        df_clean["proteins_100g"].isna()
        | df_clean["proteins_100g"].between(0, 100)
    )
    & (df_clean["salt_100g"].isna() | df_clean["salt_100g"].between(0, 100))
    & (
        df_clean["fruits-vegetables-legumes_100g"].isna()
        | df_clean["fruits-vegetables-legumes_100g"].between(0, 100)
    )
)

df_clean = df_clean[plausibel].copy()
print(f"Verbleibende plausible Produkte: {len(df_clean):,}")

Verbleibende plausible Produkte: 73,897


In [28]:
#define a function to determine the dietary form based on the ingredients_analysis_tags column


def erkenne_ernaehrungsform(tags):
    if pd.isna(tags):
        return "Unbekannt"

    # Falls tags ein String ist, in eine saubere Menge (Set) aus Einzel-Tags zerlegen
    if isinstance(tags, str):
        tag_set = set(tags.split(","))
    else:
        tag_set = set(tags)

    if "en:vegan" in tag_set:
        return "Vegan"
    if "en:vegetarian" in tag_set:
        return "Vegetarisch"
    if "en:non-vegan" in tag_set or "en:non-vegetarian" in tag_set:
        return "Nicht-vegetarisch"
    return "Unbekannt"


df_clean["diet_type"] = df_clean["ingredients_analysis_tags"].apply(
    erkenne_ernaehrungsform
)
print(df_clean["diet_type"].value_counts())

diet_type
Nicht-vegetarisch    25735
Vegan                22363
Unbekannt            17023
Vegetarisch           8776
Name: count, dtype: int64


In [29]:
df_clean[df_clean["diet_type"] == "Unbekannt"]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
7,790310020,2026-03-26T02:25:12Z,Jelly Fish,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",7.0,7.0,c,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,140.000000,0.0,0.0,34.0,24.000000,0.0,0.00,0.076200,0.00,Unbekannt
45,12200002005,2026-03-26T02:25:19Z,Chinese dumpling sheets,Master Lau,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",0.0,NaN,unknown,4.0,Cereals and potatoes,Cereals,NaN,unknown,NaN,NaN,0.7750,Pastas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,Unbekannt
49,13409918104,2026-07-28T22:01:35Z,Barbecue Sauce,Sweet Baby Ray's,"en:no-gluten,en:kosher,en:green-dot,en:orthodo...","en:austria,en:germany,en:sweden,en:united-king...","en:palm-oil-free,en:vegan-status-unknown,en:ve...",2.0,25.0,e,4.0,Fat and sauces,Dressings and sauces,70.0,b,NaN,17.0,0.7875,en:Groceries,194.444444,0.0,0.0,50.0,47.222222,0.0,0.00,2.013889,18.15,Unbekannt
51,14113910569,2026-03-25T18:16:11Z,Pistachios,Wonderful,"en:made-in-germany,en:new","en:france,en:germany","en:palm-oil-content-unknown,en:vegan-status-un...",0.0,7.0,c,NaN,Salty snacks,Nuts,28.0,e,NaN,10.0,0.7000,Salted pistachios,590.000000,46.0,5.6,17.4,7.800000,NaN,21.40,0.650000,0.00,Unbekannt
52,14113911672,2026-02-24T18:51:31Z,Pistachios,Wonderful,NaN,"en:belgium,en:france,en:germany","en:palm-oil-content-unknown,en:vegan-status-un...",0.0,10.0,c,NaN,Salty snacks,Nuts,34.0,d,NaN,15.0,0.8875,Salted pistachios,590.000000,46.0,5.6,27.7,7.800000,NaN,21.40,1.300000,0.00,Unbekannt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436438,4305615754635,2022-05-21T17:45:09Z,3 in 1 Sport Duschgel,Alterra,NaN,en:germany,"en:palm-oil-content-unknown,en:vegan-status-un...",1.0,NaN,unknown,4.0,unknown,unknown,NaN,unknown,NaN,1.0,0.7750,en:showers-and-baths,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unbekannt
436442,8005140003357,2026-05-10T11:38:12Z,Crema di Balsamico,Lanfranca,NaN,"en:belgium,en:france,en:germany",NaN,NaN,12.0,d,NaN,unknown,unknown,NaN,unknown,NaN,1.0,0.7625,en:groceries,176.000000,0.0,0.0,41.7,35.800000,NaN,0.42,0.080000,NaN,Unbekannt
436483,9005835311346,2026-03-26T19:22:02Z,Claro classic Geschirrspültabs,claro,en:eu-ecolabel,en:germany,NaN,NaN,NaN,unknown,NaN,unknown,unknown,NaN,unknown,NaN,1.0,0.7750,en:household-supplies,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unbekannt
436503,8718951249981,2026-03-25T15:06:31Z,Max White Charcoal,Colgate,NaN,en:germany,"en:palm-oil-content-unknown,en:vegan-status-un...",6.0,NaN,unknown,4.0,unknown,unknown,NaN,unknown,NaN,1.0,0.7750,en:toothpastes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,Unbekannt


In [30]:
df_clean.shape

(73897, 29)

In [31]:
# Pflichtspalten für echte Lebensmittel-Analysen definieren
pflicht_spalten = [
    "pnns_groups_1",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "proteins_100g",
    "salt_100g",
]

df_clean = df_clean.dropna(subset=pflicht_spalten).copy()

# Sicherheits-Check: Alle noch vorhandenen Kategorien anzeigen
print(df_clean["pnns_groups_1"].value_counts())
print(f"\nFinale saubere Produktanzahl: {len(df_clean):,}")

pnns_groups_1
Sugary snacks              10746
Cereals and potatoes       10012
Milk and dairy products     9533
unknown                     7949
Fish Meat Eggs              6741
Beverages                   6548
Fat and sauces              4828
Composite foods             4237
Fruits and vegetables       4227
Salty snacks                2990
Alcoholic beverages          327
Baby foods                   170
Name: count, dtype: int64

Finale saubere Produktanzahl: 68,308


In [37]:
df_clean[df_clean["pnns_groups_1"] == "unknown"]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
31,11152134079,2026-03-26T02:25:16Z,Dashi Kombu Dried Seaweed,Welpac,NaN,"en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,14.0,d,1.0,unknown,unknown,NaN,unknown,NaN,3.0,0.7875,en:undefined,200.000000,0.0,0.0,40.000000,10.000000,20.000000,0.000000,6.50000,100.00,Vegan
32,11152443683,2026-08-17T10:19:05Z,Jasmine Rice,Dynasty,"en:no-gmos,en:non-gmo-project","en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,2.0,b,1.0,unknown,unknown,NaN,unknown,NaN,1.0,0.7750,en:undefined,355.560000,0.0,0.0,80.000000,0.000000,2.220000,6.670000,0.00000,0.00,Vegan
55,14113912341,2026-03-25T18:16:16Z,Sweet Chili Pistachios,Wonderful,NaN,"en:france,en:germany","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",2.0,23.0,e,4.0,unknown,unknown,NaN,unknown,NaN,1.0,0.9875,Flavoured pistachios,574.000000,45.2,5.8,17.300000,10.300000,0.000000,20.000000,1.80000,0.10,Unbekannt
72,16229000561,2026-03-25T18:17:49Z,Lychee in Syrup,Aroy-D,"en:green-dot,fr:triman","en:australia,en:canada,en:france,en:germany","en:palm-oil-free,en:vegan-status-unknown,en:ve...",1.0,6.0,c,3.0,unknown,unknown,NaN,unknown,NaN,1.0,0.8875,Lychees in syrup,83.832335,0.0,0.0,20.359281,19.760479,0.598802,0.598802,0.07485,0.00,Unbekannt
89,17034020041,2026-03-25T18:19:06Z,Cucumbers in Brine,Beit Hashita,"en:kosher,en:no-preservatives,en:Kosher for pa...","en:france,en:germany,en:united-kingdom","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",0.0,10.0,c,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.7750,en:Cucumbers in Brine,12.000000,0.0,0.0,2.000000,0.550000,NaN,1.000000,2.50000,71.25,Unbekannt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434060,3083680001717,2026-10-12T03:58:49Z,BON Weisse Bohnen.,Bonduelle,"en:vegetarian,en:nutriscore,en:vegan,en:nutris...","en:austria,en:france,en:germany,en:netherlands","en:palm-oil-free,en:vegan,en:vegetarian",0.0,-3.0,a,3.0,unknown,unknown,NaN,unknown,NaN,4.0,0.7500,en:Vegetables - Prepared/Processed (Shelf Stable),89.000000,0.8,0.1,15.400000,0.900000,7.900000,6.600000,0.61000,26.02,Vegan
434263,7610095227799,2026-03-25T17:09:58Z,Seasoning papprika,Zweifel,"en:no-lactose,en:made-in-swiss","en:germany,en:world,en:Suisse","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",2.0,NaN,not-applicable,4.0,unknown,unknown,15.0,e,NaN,3.0,0.7625,en:Groceries,269.000000,2.8,0.0,57.000000,0.000000,0.000000,2.600000,25.00000,0.00,Unbekannt
436442,8005140003357,2026-05-10T11:38:12Z,Crema di Balsamico,Lanfranca,NaN,"en:belgium,en:france,en:germany",NaN,NaN,12.0,d,NaN,unknown,unknown,NaN,unknown,NaN,1.0,0.7625,en:groceries,176.000000,0.0,0.0,41.700000,35.800000,NaN,0.420000,0.08000,NaN,Unbekannt
436725,42512141,2026-10-30T06:25:00Z,Focaccia Mozzarella,REWE to go,"en:fsc,en:nutriscore,en:fsc-mix,en:nutriscore-...",en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",6.0,12.0,d,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.7875,de:Focaccia di Recco col formaggio,247.000000,12.0,5.6,25.000000,3.000000,1.400000,9.100000,0.86000,17.65,Nicht-vegetarisch


In [40]:
mask_sugar = df_clean["sugars_100g"] > df_clean["carbohydrates_100g"]
df_clean[mask_sugar]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
276,41321009754,2026-06-01T17:44:47Z,Balsamic Vinaigrette Dressing,Wish-Bone,"en:no-artificial-flavors,en:No HFCS","en:bolivia,en:france,en:germany,en:united-states","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",5.0,NaN,unknown,4.0,Fat and sauces,Dressings and sauces,42.0,d,NaN,8.0,0.7750,Vinaigrettes,200.0,16.666667,3.333333,9.186,13.333333,0.000000,0.0,2.416667,5.833333,Unbekannt
750,97339000054,2026-11-09T17:18:24Z,Mexican hot sauce,"Salsa Tamazula S.A. De C.V., Valentina",NaN,"en:france,en:germany,en:mexico,en:spain,en:uni...","en:palm-oil-free,en:vegan,en:vegetarian",1.0,NaN,unknown,3.0,Fat and sauces,Dressings and sauces,NaN,unknown,NaN,49.0,0.9875,en:groceries,40.0,2.000000,0.019121,2.000,3.271064,2.000000,2.0,5.400000,61.680000,Vegan
1896,20082703,2026-03-25T20:57:14Z,Emmental,Milbona,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian",0.0,NaN,unknown,3.0,Milk and dairy products,Cheese,32.0,d,NaN,31.0,0.8750,Emmentaler,373.0,29.000000,0.106700,0.100,4.782100,0.000000,28.0,1.000000,0.000000,Vegetarisch
1965,20095314,2026-03-26T00:40:05Z,Edel-Zartbitter-Schokolade Venezuela 56% Kakao,J.D. Gross,"en:vegetarian,en:vegan","en:germany,en:spain","en:palm-oil-free,en:vegan,en:vegetarian",1.0,NaN,unknown,4.0,Sugary snacks,Chocolate products,20.0,e,NaN,NaN,0.9875,Dark chocolates,551.6,39.200000,1.048661,45.200,57.316518,1.751786,4.5,0.009380,0.000000,Vegan
2537,20208646,2026-09-04T23:23:17Z,Tropical Wonder smoothie,Solevita,"en:vegetarian,en:vegan,en:green-dot,en:made-in...","en:france,en:germany","en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,22.0,e,NaN,Beverages,Unsweetened beverages,41.0,d,NaN,29.0,0.8875,Smoothies,0.0,3.200000,8.000000,11.600,31.000000,2.000000,2.0,0.037500,0.000000,Vegan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282743,4061462513503,2026-05-28T04:11:36Z,skyr BOURBON VANILLE,MILSANI,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian",0.0,NaN,unknown,4.0,Milk and dairy products,Dairy desserts,NaN,unknown,NaN,10.0,0.7000,de:Frischkäsezubereitung,76.8,0.200000,10.401500,9.500,10.911429,0.053571,8.8,0.736073,2.678571,Vegetarisch
285378,4016903000005,2026-03-26T17:53:24Z,Tofu,Soja Kang,"en:vegetarian,en:no-gmos,en:european-vegetaria...",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,3.0,Cereals and potatoes,Legumes,77.0,a,NaN,1.0,0.8000,Tofu,144.0,8.700000,1.138320,0.600,3.067605,0.600000,15.7,15.957499,41.850000,Vegan
289669,4337256837910,2026-04-10T17:54:07Z,Geriebener Mozzarella,ja!,"en:no-gmos,en:nutriscore,de:ohne-gentechnik",en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian",0.0,NaN,unknown,3.0,Milk and dairy products,Cheese,34.0,d,NaN,4.0,0.7625,Cheeses,332.0,25.000000,17.000000,3.000,4.813652,0.000000,23.0,1.180000,0.000000,Vegetarisch
303821,4061462014833,2026-03-26T18:39:38Z,Butterkäse in Scheiben,Hofburger,en:nutriscore,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",0.0,NaN,unknown,3.0,Milk and dairy products,Cheese,34.0,d,NaN,2.0,0.7875,Sliced cheeses,330.0,26.000000,17.000000,0.100,4.793994,0.000000,24.0,1.672500,0.000000,Nicht-vegetarisch


In [60]:
mask_sugar.sum()/df_clean.shape[0]*100

np.float64(0.0907653569128067)

In [61]:
df_clean = df_clean[~mask_sugar]

In [41]:
mask_fats = df_clean["saturated-fat_100g"] > df_clean["fat_100g"]
df_clean[mask_fats]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
393,55577420270,2026-03-25T20:50:14Z,Cool Blue,Gatorade,"en:kosher,en:cor-kosher","en:canada,en:germany","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",6.0,NaN,unknown,4.0,Beverages,Artificially sweetened beverages,NaN,unknown,NaN,8.0,0.7875,Dietary drink for sport,23.688663,0.000000,0.000010,6.091371,5.752961,0.000880,0.000000,0.114213,0.00,Unbekannt
900,734756000105,2026-09-05T15:10:51Z,Sticky Sweet BBQ Sauce,Stubb's,"en:no-gluten,en:kosher,en:no-gmos,en:certified...","en:australia,en:germany,en:united-states,en:world","en:palm-oil-free,en:vegan-status-unknown,en:ve...",0.0,NaN,unknown,4.0,Fat and sauces,Dressings and sauces,73.0,b,NaN,8.0,0.9000,en:Groceries,171.428571,0.000000,0.067232,37.142857,31.428571,1.903542,0.000000,1.642857,78.69,Unbekannt
2033,20108908,2026-03-23T23:47:29Z,Parsakaali,Freshona,"en:keyhole,xx:sydanmerkki","en:finland,en:france,en:germany,en:switzerland","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,Fruits and vegetables,Vegetables,99.0,a-plus,NaN,1.0,0.8000,Frozen broccolis,23.000000,0.000000,0.061000,2.000000,2.000000,2.900000,3.000000,0.048000,100.00,Vegan
2186,20138769,2026-03-14T07:30:26Z,Leichter Katensschinken,Dulano,"en:german-agricultural-society,en:gold-medal-o...","en:france,en:germany","en:palm-oil-free,en:non-vegan,en:non-vegetarian",2.0,NaN,unknown,4.0,Fish Meat Eggs,Processed meat,17.0,e,NaN,2.0,1.0000,en:Zubereitetes Fleisch,83.000000,2.100000,3.519340,0.500000,0.003119,0.000000,15.800000,0.006250,0.00,Nicht-vegetarisch
2537,20208646,2026-09-04T23:23:17Z,Tropical Wonder smoothie,Solevita,"en:vegetarian,en:vegan,en:green-dot,en:made-in...","en:france,en:germany","en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,22.0,e,NaN,Beverages,Unsweetened beverages,41.0,d,NaN,29.0,0.8875,Smoothies,0.000000,3.200000,8.000000,11.600000,31.000000,2.000000,2.000000,0.037500,0.00,Vegan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317677,4061462055959,2026-11-15T18:49:13Z,Bio-agavendicksaft,"Aldi, ALDI Nord BIO","en:vegetarian,en:organic,en:eu-organic,en:euro...",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,unknown,unknown,46.0,c,NaN,11.0,0.7875,Agave syrups,304.000000,0.000000,0.265475,76.000000,73.000000,0.000000,0.000000,0.000000,0.00,Vegan
325680,4061461664589,2026-05-09T10:08:18Z,Bio-Orangensaft,"Aldi, Aldi Bio","en:vegetarian,en:organic,en:eu-organic,en:nutr...",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,Beverages,Fruit juices,60.0,b,NaN,13.0,1.0000,Orange juices,41.000000,0.000000,0.053000,8.800000,8.800000,0.280000,0.700000,0.000000,100.00,Vegan
327367,4062139025886,2026-08-28T17:27:25Z,Pfirsich Ice Tea Zero Zucker,Lipton,"en:low-or-no-sugar,en:no-sugar,en:rainforest-a...",en:germany,"en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",5.0,NaN,unknown,4.0,Beverages,Artificially sweetened beverages,95.0,a-plus,NaN,3.0,0.8000,Iced teas,0.400000,0.000000,0.000007,0.000000,0.000000,0.000000,0.000000,0.012000,0.10,Unbekannt
333500,4066600208402,2026-09-14T14:01:53Z,Limo Zitrone,Paulaner,"en:vegetarian,en:no-gluten,en:vegan",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",2.0,NaN,unknown,3.0,Beverages,Sweetened beverages,NaN,not-applicable,NaN,7.0,0.7875,Lemon soft drinks,34.000000,0.000000,0.001511,8.300000,8.300000,0.014125,0.000000,0.000000,5.00,Vegan


In [59]:
mask_fats.sum()/df_clean.shape[0]*100

np.float64(0.13907595010833285)

In [64]:
df_clean = df_clean[~mask_fats]

/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_27604/1015690435.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_clean = df_clean[~mask_fats]


In [ ]:
mask_macros = df_clean["fat_100g"] + df_clean["carbohydrates_100g"] + df_clean["proteins_100g"] > 101
df_clean[mask_macros]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type,absolute_calories_difference,theoretical_calories_difference
1013,884912129512,2026-10-13T14:19:35Z,Sweetened Naturally & Artificially Flavored Ri...,Cocoa Pebbles,en:no-gluten,"en:germany,en:mexico,en:united-states","en:palm-oil-free,en:vegan-status-unknown,en:ve...",2.0,33.0,e,4.0,Cereals and potatoes,Breakfast cereals,52.0,c,NaN,8.0,0.8875,Puffed cereals,490.555556,9.444444,6.666667,86.666667,46.388889,1.111111,14.722222,1.671528,NaN,Unbekannt,4.547474e-13,-4.547474e-13
2893,20332662,2026-03-25T18:24:46Z,Corn flakes 850g,Crownfield,"en:german-agricultural-society,en:gold-medal-o...","en:denmark,en:germany,en:switzerland","en:palm-oil-free,en:vegan-status-unknown,en:ve...",0.0,0.0,a,3.0,Cereals and potatoes,Breakfast cereals,55.0,c,NaN,10.0,0.9500,Corn flakes,494.600000,4.600000,2.600000,96.000000,17.600000,3.400000,15.600000,1.240000,NaN,Unbekannt,6.800000e+00,-6.800000e+00
3966,20884291,2026-10-30T12:05:51Z,Knusper,"Crownfield, Brüggen, Lidl","en:low-or-no-sugar,en:low-sugar,en:reduced-sug...","en:france,en:germany","en:palm-oil-content-unknown,en:non-vegan,en:ve...",1.0,15.0,d,4.0,Cereals and potatoes,Breakfast cereals,65.0,b,NaN,59.0,0.9000,Mueslis,546.000000,16.000000,3.800000,77.400000,23.000000,7.400000,19.600000,0.980000,NaN,Nicht-vegetarisch,1.400000e+01,-1.400000e+01
7207,3360101880107,2026-03-17T09:33:54Z,Etui 18g rose,Les Anis De Flavigny,"en:vegetarian,en:vegan,en:eve-vegan","en:belgium,en:france,en:germany,en:italy,en:mo...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,20.0,e,4.0,Sugary snacks,Sweets,72.0,b,NaN,1.0,0.7875,Candies,398.000000,4.000000,0.000000,99.000000,97.000000,0.000000,0.000000,0.000000,0.000000,Vegan,3.400000e+01,3.400000e+01
13926,4016249696122,2026-03-26T01:51:24Z,Agavendicksaft,Allos,"en:organic,en:vegetarian,en:eu-organic,en:vega...",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,20.0,e,NaN,unknown,unknown,52.0,c,NaN,1.0,0.9875,Agave syrups,420.000000,1.000000,0.100000,100.000000,100.000000,NaN,1.000000,0.100000,0.000000,Vegan,7.000000e+00,-7.000000e+00
24938,5900020016249,2026-11-08T23:09:39Z,Cini minis,Nestlé,NaN,"en:czech-republic,en:germany,en:hungary","en:palm-oil,en:vegan-status-unknown,en:vegetar...",2.0,29.0,e,4.0,Cereals and potatoes,Breakfast cereals,45.0,c,NaN,1.0,0.7875,Extruded cereals,620.333333,17.000000,5.000000,95.000000,44.333333,4.333333,19.666667,1.466667,NaN,Unbekannt,8.666667e+00,-8.666667e+00
44549,8697412872428,2026-03-25T19:30:37Z,Ahinur Haselnuss Kakao Creme,Ahinur,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",1.0,18.0,d,4.0,Sugary snacks,Sweets,NaN,unknown,NaN,NaN,0.7875,Sweet spreads,538.000000,36.170000,0.000000,59.980000,51.770000,14.570000,5.940000,0.230000,0.000000,Nicht-vegetarisch,5.121000e+01,5.121000e+01
47657,4316268565295,2026-11-10T22:34:25Z,Pinky Donuts 6er-Pack,Backstube,NaN,en:germany,"en:palm-oil,en:non-vegan,en:non-vegetarian",8.0,34.0,e,4.0,Sugary snacks,Biscuits and cakes,67.0,b,NaN,1.0,0.7875,Doughnuts,652.580645,33.870968,20.967742,79.032258,27.419355,NaN,7.903226,1.612903,NaN,Nicht-vegetarisch,1.136868e-13,-1.136868e-13
61865,3360102110012,2026-03-19T18:43:44Z,Etui 18g anis bio,Les Anis de Flavigny®,"en:vegetarian,en:organic,en:vegan,en:eve-vegan","en:belgium,en:france,en:germany,en:italy,en:mo...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,20.0,e,4.0,Sugary snacks,Sweets,72.0,b,NaN,1.0,0.7750,Candies,398.000000,3.000000,0.000000,99.000000,97.000000,0.000000,0.000000,0.000000,0.000000,Vegan,2.500000e+01,2.500000e+01
65502,4260014351550,2026-02-

In [58]:
mask_macros.sum()/df_clean.shape[0]*100

np.float64(0.023423317912982376)

In [63]:
df_clean = df_clean[~mask_macros]

/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_27604/743571433.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_clean = df_clean[~mask_macros]


In [46]:
mask_calories = ((4 * (df_clean["carbohydrates_100g"] + df_clean["proteins_100g"]) + 9 * df_clean["fat_100g"]) - df_clean["energy-kcal_100g"]).abs() / df_clean["energy-kcal_100g"] > 0.2
df_clean[mask_calories].head(50)

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
100,21000067060,2026-06-09T15:12:28Z,mac & cheese original flavor,Kraft,"en:no-artificial-flavors,en:no-artificial-pres...","en:bolivia,en:france,en:germany,en:united-kingdom","en:palm-oil-free,en:non-vegan,en:maybe-vegetarian",4.0,20.0,e,4.0,Cereals and potatoes,Cereals,NaN,unknown,NaN,9.0,0.8875,Durum wheat macaroni,514.285714,4.232875,2.116438,66.315048,8.465751,2.821917,12.698626,2.010616,0.000000,Nicht-vegetarisch
160,27000500118,2026-08-20T14:28:53Z,Pasta Sauce Garlic & Herb,Hunt's,"en:vegetarian,en:vegan","en:germany,en:spain,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",2.0,3.0,c,4.0,Fat and sauces,Dressings and sauces,70.0,b,NaN,3.0,0.7000,en:Groceries,28.000000,0.000000,0.000000,4.480000,2.560000,0.640000,0.640000,0.960000,57.130000,Vegan
294,41500888200,2026-11-10T13:54:40Z,Red Hot Wings Buffalo Sauce,Frank’s,NaN,"en:france,en:germany,en:united-kingdom","en:palm-oil-free,en:vegan-status-unknown,en:ve...",1.0,20.0,e,4.0,Fat and sauces,Dressings and sauces,NaN,unknown,NaN,10.0,0.9000,Hot sauces,31.000000,1.400000,0.100000,2.300000,0.000000,0.800000,0.700000,5.685000,1.310000,Unbekannt
388,54400000054,2026-06-22T17:55:19Z,Original Sauce,A.1.,NaN,"en:france,en:germany,en:united-states","en:palm-oil-free,en:vegan-status-unknown,en:ve...",3.0,22.0,e,4.0,Fat and sauces,Dressings and sauces,65.0,b,NaN,5.0,0.8000,en:Groceries,88.235294,0.000000,0.000000,17.647059,11.764706,NaN,0.000000,4.264706,40.530000,Unbekannt
430,66200912734,2026-03-14T02:58:56Z,Montreal Chicken seasoning,"club house, null",en:halal,"en:france,en:germany","en:palm-oil-free,en:vegan-status-unknown,en:ve...",1.0,0.0,a,4.0,unknown,unknown,NaN,unknown,NaN,3.0,0.7000,Condiments,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.100000,0.000000,28.125000,Unbekannt
439,670272,2026-10-26T12:50:26Z,Bolognese Pasta Sauce,"Marks & Spencers, Marks & Spencer","en:green-dot,en:made-in-italy","en:germany,en:united-kingdom","en:palm-oil-free,en:non-vegan,en:non-vegetarian",1.0,-2.0,a,3.0,Fat and sauces,Dressings and sauces,17.0,e,NaN,1.0,0.8875,en:Groceries,122.000000,4.700000,1.800000,5.100000,2.700000,1.200000,5.200000,0.750000,84.820000,Nicht-vegetarisch
443,70177010652,2026-07-20T09:49:01Z,Earl Grey Tea,Twinings,fr:Ethical Tea Partership,"en:germany,en:switzerland","en:palm-oil-free,en:vegan-status-unknown,en:ve...",0.0,0.0,b,4.0,Beverages,Teas and herbal teas and coffees,84.0,a,NaN,2.0,0.7875,Teas,1.000000,0.000000,0.000000,0.200000,0.000000,NaN,0.100000,0.000000,0.000000,Unbekannt
445,70177010775,2026-03-25T23:09:58Z,Tea English Schwarztee,Twinings,en:green-dot,"en:belgium,en:germany,en:italy,en:switzerland","en:palm-oil-free,en:vegan,en:vegetarian",0.0,NaN,unknown,1.0,Beverages,Teas and herbal teas and coffees,84.0,a,NaN,9.0,0.8750,Black teas,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Vegan
451,70177068011,2026-04-04T13:34:26Z,Vanilla Tea,"Twinings, Twinnings of London",NaN,"en:germany,en:switzerland","en:palm-oil-free,en:maybe-vegan,en:maybe-veget...",0.0,0.0,b,4.0,Beverages,Teas and herbal teas and coffees,80.0,a,NaN,2.0,0.8750,Vanilla teas,1.000000,0.000000,0.000000,0.200000,0.000000,NaN,0.100000,0.000000,0.000000,Unbekannt
473,70573555559,2026-03-25T20:32:28Z,Okra Pickles,Talk O' Texas,NaN,"en:germany,en:united-states","en:may-contain-palm-oil,en:maybe-vegan,en:mayb...",1.0,16.0,d,4.0,unknown,unknown,NaN,unknown,NaN,1.0,0.8750,en:Salted snacks,17.857143,0.000000,0.000000,3.571429,0.000000,0.000000,0.000000,3.482143,57.142857,Unbekannt


In [65]:
mask_calories.sum()

np.int64(2706)

In [51]:
df_clean["theoretical_calories_difference"] = ((4 * (df_clean["carbohydrates_100g"] + df_clean["proteins_100g"]) 
                                                + 9 * df_clean["fat_100g"]) 
                                                - df_clean["energy-kcal_100g"])

In [ ]:
df_clean.head()

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type,absolute_calories_difference,theoretical_calories_difference
6,790310013,2026-03-26T02:25:12Z,Sour Fruit Gummies,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:non-vegetarian",9.0,19.0,e,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,325.000000,0.000000,0.0,75.000000,57.500000,0.0,5.000000,0.1270,11.363636,Nicht-vegetarisch,5.000000,-5.000000
7,790310020,2026-03-26T02:25:12Z,Jelly Fish,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",7.0,7.0,c,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,140.000000,0.000000,0.0,34.000000,24.000000,0.0,0.000000,0.0762,0.000000,Unbekannt,4.000000,-4.000000
17,9800895007,2026-07-17T18:42:50Z,Nutella,Nutella,en:no-gluten,"en:germany,en:united-states","en:palm-oil,en:non-vegan,en:vegetarian-status-...",0.0,23.0,e,4.0,Sugary snacks,Sweets,19.0,e,NaN,67.0,0.8000,Cocoa and hazelnuts spreads,540.000000,29.730000,10.8,59.460000,56.760000,2.7,5.410000,0.1000,0.000000,Nicht-vegetarisch,12.950000,-12.950000
22,10700021526,2026-09-20T19:46:51Z,Milk Duds Candy,Milk Duds,"en:kosher,en:orthodox-union-kosher","en:germany,en:united-states","en:palm-oil,en:non-vegan,en:vegetarian-status-...",4.0,32.0,e,4.0,Sugary snacks,Chocolate products,72.0,b,NaN,9.0,0.7875,Candies,433.333333,16.666667,10.0,73.333333,56.666667,NaN,1.666667,0.6250,0.000000,Nicht-vegetarisch,16.666667,16.666667
31,11152134079,2026-03-26T02:25:16Z,Dashi Kombu Dried Seaweed,Welpac,NaN,"en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,14.0,d,1.0,unknown,unknown,NaN,unknown,NaN,3.0,0.7875,en:undefined,200.000000,0.000000,0.0,40.000000,10.000000,20.0,0.000000,6.5000,100.000000,Vegan,40.000000,-40.000000


In [66]:
df_clean.shape

(68143, 31)

In [69]:
df_clean.to_csv("../data/processed/df_clean.csv", index=False)